In [30]:
import torch 
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
''' 
1.5B 모델 선택의 이유 :
8GB VRAM 중 3GB VRAM을 차지
나머지 5GB는 
추론 과정에서 문맥 길이에 비례해 늘어나는 KV Cache를 위해 남겨둠
학습 과정에서 역전파를 위한 기울기과 옵티마이저 상태를 저장하는 데 사용
'''

' \n1.5B 모델 선택의 이유 :\n8GB VRAM 중 3GB VRAM을 차지\n나머지 5GB는 \n추론 과정에서 문맥 길이에 비례해 늘어나는 KV Cache를 위해 남겨둠\n학습 과정에서 역전파를 위한 기울기과 옵티마이저 상태를 저장하는 데 사용\n'

In [31]:
# start with a Hugging Face tokenizer / BPE와 subword tokenization

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Byte-Pair Encoding 모델을 사용하고 있다!
print(tokenizer.backend_tokenizer.model)

text = "He wanted to eat Chinese food and behaveed uncharracteristically."

tokens = tokenizer.tokenize(text)

ids = tokenizer.encode(text)

print(tokens)
print(ids)

for tk_id in ids:
    print(
        tk_id,
        tokenizer.decode([tk_id])
    )

'''
1. 서브 워드 분해 ( 오타단어와 긴 파생어 처리 )

2. 특수 기호 Ġ의 의미 ( 단어 앞에 붙은 공백을 의미 )
'''

    

BPE(dropout=None, unk_token=None, continuing_subword_prefix="", end_of_word_suffix="", fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={"!":0, """:1, "#":2, "$":3, "%":4, ...}, merges=[("Ġ", "Ġ"), ("ĠĠ", "ĠĠ"), ("i", "n"), ("Ġ", "t"), ("ĠĠĠĠ", "ĠĠĠĠ"), ...])
['He', 'Ġwanted', 'Ġto', 'Ġeat', 'ĠChinese', 'Ġfood', 'Ġand', 'Ġbehave', 'ed', 'Ġun', 'ch', 'arr', 'acter', 'istically', '.']
[1519, 4829, 311, 8180, 8453, 3607, 323, 35692, 291, 650, 331, 1118, 2481, 37110, 13]
1519 He
4829  wanted
311  to
8180  eat
8453  Chinese
3607  food
323  and
35692  behave
291 ed
650  un
331 ch
1118 arr
2481 acter
37110 istically
13 .


'\n1. 서브 워드 분해 ( 오타단어와 긴 파생어 처리 )\n\n2. 특수 기호 Ġ의 의미 ( 단어 앞에 붙은 공백을 의미 )\n'

In [32]:
# 입력 텐서 만들기

inputs = tokenizer(
    text,
    return_tensors="pt"
)
# tokenizer(text,return_tensors="pt") -> 토큰화 + 정수 ID 변환 + 어텐션 마스크 생성 + 패딩 + 텐서 변환을 해줌

print(inputs["input_ids"])
print(inputs["input_ids"].shape, "\n\n")

print(inputs["attention_mask"]) # 모델이 어떤 토큰에 주의를 기울여야 하는 지 알려주는 마스크 ( 1은 실제 유효한 토큰, 0은 길이 맞춤용 패딩 토큰 )
print(inputs["attention_mask"].shape, "\n\n")

print(inputs)

tensor([[ 1519,  4829,   311,  8180,  8453,  3607,   323, 35692,   291,   650,
           331,  1118,  2481, 37110,    13]])
torch.Size([1, 15]) 


tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
torch.Size([1, 15]) 


{'input_ids': tensor([[ 1519,  4829,   311,  8180,  8453,  3607,   323, 35692,   291,   650,
           331,  1118,  2481, 37110,    13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [33]:
# Load the Real Causal model - Qwen2.5-1.5B의 학습된 가중치를 GPU에 올리기

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,  # 
    device_map="auto"
)
# dtype=torch.float16 : 기존 32bit에서 16bit 반정밀도로 가중치를 로드하여, 정확도 손실은 최소화하면서 GPU VRAM 사용량을 절반으로 줄이고 연산 속도를 높임! 
# device_map="auto" 의 역할 : accelerator lib를 통해서 현재 하드웨어를 자동으로 감지하고, 수동으로 to Cuda 호출 필요없이 GPU에 최적 배치

model.eval() 

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

```text
Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)   // Vocab_size와 embedding_vector
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer( // 28개의 layer x 디코더 only 구조 확인 가능
        (self_attn): Qwen2Attention( // Q, K, V, O 확인가능 -> GQA가 적용됨!
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False) // 어텐션 연산 결과를 모델 차원(1536)으로 최종 출력
        )
        (mlp): Qwen2MLP( // Feed Forwarding 부분
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation() // 활성함수 
        )
        // 층정규화 부분
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06) 
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbedding()
  )
  (lm_head): Linear(in_features=1536, out_features=151936, bias=False) // 은닉 벡터(1536)를 다시 전체 어휘(151936)의 로짓(점수)으로 바꾸는 출력층
)
```

```
(self_attn): Qwen2Attention( 
  (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
  (k_proj): Linear(in_features=1536, out_features=256, bias=True)
  (v_proj): Linear(in_features=1536, out_features=256, bias=True)
  (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
)
```
왜 out_features가 서로 다른 값을 가지고 있을까?

이 모델이 GQA (Grouped-Query Attention)를 쓰기 때문 GQA 그게 모야.... -> <https://devocean.sk.com/blog/techBoardDetail.do?ID=165192>

q_proj는 1536 차원으로 투영하지만, k_proj와 v_proj는 256 차원으로 줄여서 만든다.
쿼리 헤드 여러 개가 Key/Value 헤드 한 쌍을 공유하는 방식이라 성능 손실은 최소화하면서 속도와 메모리 효율 챙기기 위해서
그리고 o_proj는 다시 1536으로 돌아오는데, 이는 어텐션 연산 이후 잔차 연결(residual connection)이나 MLP 입력 차원인 hidden_size = 1536과 맞추기 위해서

In [34]:
# 입력 텐서들을 GPU로 이동
inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}


# find out the logits

with torch.no_grad():
    outputs = model(**inputs)
    
logits = outputs.logits

print("input_ids:", inputs["input_ids"].shape)
print("logits:", logits.shape)

'''
logits : (batch_size, sequence_length, vocab_size)
'''

input_ids: torch.Size([1, 15])
logits: torch.Size([1, 15, 151936])


'\nlogits : (batch_size, sequence_length, vocab_size)\n'

In [35]:
# Extract the next_logit
next_logits = logits[0, -1] # 마지막 토큰 위치의 logit에서 
print("next_logits:", next_logits.shape)

# convert logits to probabilities and topK - 상위 10개 다음 토큰을 추출하기
probs = torch.softmax(next_logits, dim=-1)
top_probs, top_ids = torch.topk(probs, k = 10)
for prob, tk_id in zip(top_probs, top_ids):
    token = tokenizer.decode([tk_id.item()])
    print(repr(token), f"{prob.item():4f}")

next_logits: torch.Size([151936])
' ' 0.602539
' He' 0.067627
' What' 0.032196
' A' 0.020615
' Choose' 0.018341
' Which' 0.014847
' The' 0.012024
' This' 0.010208
' It' 0.009224
' How' 0.008530
